In [25]:
import cv2
import numpy as np

from pathlib import Path

import json

import matplotlib.pyplot as plt

In [26]:
# ======================================================
# Load Project State
# ======================================================

ROOT = Path.cwd().parent


STATE_FILE = ROOT / "config" / "project_state.json"



if not STATE_FILE.exists():

    raise Exception(
        """
        project_state.json not found.

        Please run:
        01_Project_Setup.ipynb first
        """
    )



with open(
    STATE_FILE,
    encoding="utf-8"
) as f:

    PROJECT_STATE=json.load(f)



PROJECT_STATE

Exception: 
        project_state.json not found.

        Please run:
        01_Project_Setup.ipynb first
        

In [ ]:
# ======================================================
# Path
# ======================================================


INPUT_FILE = Path(
    PROJECT_STATE["input_file"]
)



PROCESSED_DIR = ROOT / "data" / "processed"


PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)



print(
INPUT_FILE
)

In [ ]:
# ======================================================
# Load Image
# ======================================================


def load_image(path):


    img=cv2.imread(
        str(path)
    )


    if img is None:

        raise Exception(
            "Image loading failed"
        )


    img=cv2.cvtColor(

        img,

        cv2.COLOR_BGR2RGB

    )


    return img

In [ ]:
def show(img,title):


    plt.figure(
        figsize=(10,10)
    )


    plt.imshow(
        img,
        cmap="gray"
    )


    plt.title(title)


    plt.axis(
        "off"
    )


    plt.show()

In [ ]:
original = load_image(
    INPUT_FILE
)


show(
    original,
    "Original"
)

In [ ]:
def grayscale(img):

    return cv2.cvtColor(

        img,

        cv2.COLOR_RGB2GRAY

    )



gray = grayscale(
    original
)


show(
    gray,
    "Gray"
)

In [ ]:
def remove_noise(img):

    return cv2.fastNlMeansDenoising(

        img,

        None,

        10,

        7,

        21

    )



clean = remove_noise(
    gray
)


show(
    clean,
    "Noise Removed"
)

In [ ]:
def enhance(img):


    clahe=cv2.createCLAHE(

        clipLimit=3,

        tileGridSize=(8,8)

    )


    return clahe.apply(
        img
    )



enhanced=enhance(
    clean
)


show(
    enhanced,
    "Enhanced"
)

In [ ]:
def threshold(img):


    return cv2.adaptiveThreshold(

        img,

        255,

        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,

        cv2.THRESH_BINARY,

        31,

        5

    )



binary=threshold(
    enhanced
)


show(
    binary,
    "Binary"
)

In [ ]:
def detect_angle(img):


    coords=np.column_stack(

        np.where(img < 255)

    )


    angle=cv2.minAreaRect(
        coords
    )[-1]


    if angle < -45:

        angle=-(90+angle)

    else:

        angle=-angle


    return angle



angle=detect_angle(
    binary
)


print(
"Angle:",
angle
)

In [ ]:
def rotate_image(img,angle):


    h,w=img.shape


    center=(w//2,h//2)


    matrix=cv2.getRotationMatrix2D(

        center,

        angle,

        1

    )


    return cv2.warpAffine(

        img,

        matrix,

        (w,h),

        flags=cv2.INTER_CUBIC,

        borderMode=cv2.BORDER_REPLICATE

    )



deskew=rotate_image(

    binary,

    angle

)


show(
    deskew,
    "Deskew"
)

In [ ]:
edges=cv2.Canny(

    deskew,

    50,

    150

)


show(
    edges,
    "Edges"
)

In [ ]:
def quality(edge):

    return round(

        np.count_nonzero(edge)

        /

        edge.size

        *

        100,

        3

    )


QUALITY = quality(
    edges
)


print(

"Quality:",

QUALITY,

"%"

)

In [ ]:
cv2.imwrite(

    str(
        PROCESSED_DIR /
        "processed.png"
    ),

    deskew

)


cv2.imwrite(

    str(
        PROCESSED_DIR /
        "edges.png"
    ),

    edges

)


print(
"Saved"
)

In [ ]:
# ======================================================
# Update State
# ======================================================


PROJECT_STATE.update({

    "processed_image":
    str(
        PROCESSED_DIR /
        "processed.png"
    ),

    "edge_image":
    str(
        PROCESSED_DIR /
        "edges.png"
    ),

    "quality":
    QUALITY

})



with open(

STATE_FILE,

"w",

encoding="utf-8"

) as f:


    json.dump(

        PROJECT_STATE,

        f,

        indent=4

    )



print(
"State Updated"
)